# Validation — best-of-N and self-consistency

On ambiguous questions a single pass can confidently give the *wrong* answer. Aura's
validation extension generates N candidates and picks the best:

- **best_of_n** — generate N responses, select by criteria (`HighestConfidence`, `Longest`, `MostRelevant`, `Shortest`)
- **self_consistency** — generate N responses, require agreement above `min_confidence`
- **confidence_threshold** — reject low-confidence responses

The response carries a `validation` block describing what happened (strategy, candidates
generated, selected index, confidence).

In [ ]:
from aura import AuraClient

client = AuraClient()

In [ ]:
# A question where naive sampling disagrees:
# "All but 9 run away" means 9 remain.
ambiguous = (
    "A farmer has 17 sheep. All but 9 run away. "
    "How many are left? Answer with just the number."
)

response = client.responses.create(
    model="gpt-5.4-mini",
    input=ambiguous,
    validation={
        "strategy": "best_of_n",
        "n": 3,
        "selection": "HighestConfidence",
    },
)

print(f"answer:            {response.output_text}")
v = response.validation
if v:
    print(f"strategy:          {v.strategy.value if v.strategy else '—'}")
    print(f"candidates:        {v.candidates_generated}")
    print(f"selected index:    {v.selected_index}")
    print(f"confidence:        {v.confidence}")
else:
    print("no validation metadata returned")

In [ ]:
# self_consistency: 3 candidates must agree at >= 0.7 confidence
response2 = client.responses.create(
    model="gpt-5.4-mini",
    input=ambiguous,
    validation={
        "strategy": "self_consistency",
        "n": 3,
        "min_confidence": 0.7,
    },
)

print(f"answer:            {response2.output_text}")
v2 = response2.validation
if v2:
    print(f"strategy:          {v2.strategy.value if v2.strategy else '—'}")
    print(f"candidates:        {v2.candidates_generated}")
    print(f"confidence:        {v2.confidence}")
    print(f"min_confidence:    {v2.min_confidence}")
else:
    print("no validation metadata returned")

Both strategies burn N generations per call — that's the token cost of confidence.
Use them where a wrong answer is expensive (classification, extraction, grading) and skip
them for casual chat.

Next: [06 — feedback few-shot](06_feedback_few_shot.ipynb).